# Measure Gabriel's horn: finite volume, unbounded area

*Adversarial Mathematics From First Principles*, [chapter 9](https://programmer.ie/books/math/09-chapter/) · [worked example](https://programmer.ie/examples/math/09-chapter/)

Run the cells in order. Everything below the program is taken from the laboratory's recorded, hash-verified run; the program is a small, self-contained demonstration of the same mathematics.

Rotate y = 1/x, for x ≥ 1, about the x-axis. The solid is infinitely long, yet
its volume is finite, while its surface area is infinite. This program
integrates both up to b = 10, 100, …, 10^6 and compares them with their exact
forms, π(1 − 1/b) and the lower bound 2π ln b.

## Run it

Standard library only; nothing to install.

In [1]:
"""Gabriel's horn: y = 1/x for x >= 1, rotated about the x-axis (standard library only).

Volume up to b:   V(b) = integral_1^b pi / x^2 dx = pi (1 - 1/b)   -> pi
Area up to b:     A(b) = integral_1^b 2 pi (1/x) sqrt(1 + 1/x^4) dx >= 2 pi ln b -> infinity
"""
import math


def integrate(f, a, b, steps=20000):
    """Simpson's rule in the variable u = ln x, which suits a horn this long."""
    lo, hi = math.log(a), math.log(b)
    h = (hi - lo) / steps
    s = f(a) * a + f(b) * b
    for k in range(1, steps):
        x = math.exp(lo + k * h)
        s += (4 if k % 2 else 2) * f(x) * x
    return s * h / 3


volume = lambda x: math.pi / x ** 2
area = lambda x: 2 * math.pi / x * math.sqrt(1 + x ** -4)

print("       b      V(b)          pi(1 - 1/b)    A(b)          2 pi ln b")
for k in range(1, 7):
    b = 10.0 ** k
    print(f"{b:10.0e}  {integrate(volume, 1, b):.10f}  {math.pi * (1 - 1 / b):.10f}  "
          f"{integrate(area, 1, b):12.6f}  {2 * math.pi * math.log(b):12.6f}")
print()
print(f"pi = {math.pi:.10f}: the volume approaches it; the area passes every bound.")
print()
# Failure control: a wrong volume (say 2 pi) is contradicted by the computed values.
print(f"control: V(1e6) = {integrate(volume, 1, 1e6):.6f}, not 2 pi = {2 * math.pi:.6f}")

       b      V(b)          pi(1 - 1/b)    A(b)          2 pi ln b
     1e+01  2.8274333882  2.8274333882     15.177450     14.467569
     1e+02  3.1101767271  3.1101767271     29.645097     28.935138
     1e+03  3.1384510609  3.1384510609     44.112666     43.402706
     1e+04  3.1412784943  3.1412784943     58.580235     57.870275
     1e+05  3.1415612377  3.1415612377     73.047804     72.337844
     1e+06  3.1415895120  3.1415895120     87.515373     86.805413

pi = 3.1415926536: the volume approaches it; the area passes every bound.

control: V(1e6) = 3.141590, not 2 pi = 6.283185


## What the output shows

The volume settles on π; the area keeps growing, always above 2π ln b, which
has no limit. Numbers at six points prove neither statement: the convergence
and the divergence are the Lean theorems. "Fill but cannot paint" is a
statement about these two integrals only; physical paint has thickness.

## What the laboratory checked

The recorded run `20261113T233641Z-bf73f1ee` ([evidence](https://github.com/ernanhughes/math/tree/main/experiments/investigations/p08-gabriel/runner-runs/2026-11-13-v2-pinned)) is the book's result; the program above is a reader-sized demonstration of the same mathematics. **Evidence:** Fresh kernel check (Lean with Mathlib, one kernel) plus symbolic and numerical computation. **Status:** `kernel_checked_correspondence_pending`.

> Let H be the solid obtained by rotating y = 1/x, x ≥ 1, about the x-axis. Its volume ∫₁^∞ π (1/x)² dx is finite and equals π, while its surface area ∫₁^∞ 2π (1/x) √(1 + ((1/x)′)²) dx diverges.

| Check | System | Recorded outcome |
| --- | --- | --- |
| `lean` | Lean kernel | 6 of 6 declarations accepted and bound to their pinned statements |
| `compute` | Python + SymPy (symbolic integration) and mpmath (numerical check) | exit 0, verdict horn_as_claimed |

Failure controls: **5 of 5** rejected or refused as required (`lean-wrong-volume`, `lean-wrong-truncation`, `lean-area-bound-too-large`, `lean-binding-wrong-statement`, `compute-wrong-volume`).
Correspondence `P08-CORR-01` (does the formal statement say what the claim means?): **pending the author's approval**.
Contract `P08-GABRIEL-01` version 2, sha256 `9732bda4749f208d…`; `result.json` sha256 `aace58500d0a6c30…`.

## Not shown

As the laboratory's own record states:

- That these integrals are the volume and surface area of the solid: that is the classical formula (P08-B01), assumed rather than derived.
- Anything about Torricelli's own figure or proof, until the primary text is transcribed: his solid is truncated and compared with a cylinder, and he did not state the infinite area.
- Any 'painter's paradox': physical paint has thickness, and nothing physical is modelled.
- A Rocq check: this investigation is single-kernel.

## Reproduce or challenge it

To reproduce the laboratory's run, build the `math-env` image from a clone of the repository ([Getting Started](/books/math/29-chapter/)) and run:

```bash
img=$(docker image inspect math-env --format '{{.Id}}')
docker run --rm --network none -e UV_OFFLINE=1 -e MATHENV_IMAGE_ID=$img -v "${PWD}:/source:ro" -v "${PWD}/out:/out" math-env uv run --offline math-engine proof gabriel --out /out/proof-gabriel
```

This Docker form was tested for `fermat`, `kempe` and `simpson`, not separately for `gabriel`. To re-verify the recorded run itself without Docker, `math-engine proof --verify experiments/investigations/p08-gabriel/runner-runs/2026-11-13-v2-pinned` (R-V08).

Examine the two integrals. Do the Lean statements say "converges to π" and
"diverges", with the integrand and domain exactly as stated? Is the area
formula applied correctly to f(x) = 1/x? Then compare the modern horn with
Torricelli's truncated solid of 1643, and propose a formal statement closer to
his own theorem.